# Neural Networks & Backprop from Scratch

A 2-layer MLP in pure NumPy with full manual backpropagation. This is the highest-signal ML coding question: it tests chain-rule calculus, vectorized gradient computation, gradient checking, and debugging intuition. Every major tech company has some variant of this.

## What Interviewers Test
- Forward pass: matrix shapes through linear + activation layers
- Backprop: chain rule applied to each layer, correct use of transpose
- Gradient checking: finite differences as a correctness oracle
- Training loop: mini-batch, weight updates, loss tracking
- Common bugs: missing 1/m, wrong transpose, forgetting to zero gradients
- Decision boundary / convergence verification

## 2-Layer MLP Forward Pass

Architecture: $X \xrightarrow{W_1, b_1} Z_1 \xrightarrow{\text{ReLU}} A_1 \xrightarrow{W_2, b_2} Z_2 \xrightarrow{\text{Softmax}} \hat{y}$

Shapes (batch size $n$, input dim $d$, hidden dim $h$, classes $C$):
- $Z_1 = XW_1 + b_1$: $(n,d)\cdot(d,h)+(h,) \rightarrow (n,h)$
- $A_1 = \text{ReLU}(Z_1)$: $(n,h)$
- $Z_2 = A_1 W_2 + b_2$: $(n,h)\cdot(h,C)+(C,) \rightarrow (n,C)$
- $\hat{y} = \text{softmax}(Z_2)$: $(n,C)$


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
np.random.seed(42)

def relu(z):
    return np.maximum(0, z)

def relu_grad(z):
    return (z > 0).astype(float)

def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    exp_z = np.exp(z)
    return exp_z / exp_z.sum(axis=1, keepdims=True)

def cross_entropy_loss(y_hat, y_onehot):
    eps = 1e-12
    n = y_hat.shape[0]
    return -np.sum(y_onehot * np.log(y_hat + eps)) / n

class MLP:
    def __init__(self, input_dim, hidden_dim, output_dim, lr=0.01):
        # He init for ReLU
        self.W1 = np.random.randn(input_dim, hidden_dim) * np.sqrt(2/input_dim)
        self.b1 = np.zeros(hidden_dim)
        self.W2 = np.random.randn(hidden_dim, output_dim) * np.sqrt(2/hidden_dim)
        self.b2 = np.zeros(output_dim)
        self.lr = lr
        self.cache = {}  # store intermediates for backprop

    def forward(self, X):
        Z1 = X @ self.W1 + self.b1          # (n, h)
        A1 = relu(Z1)                        # (n, h)
        Z2 = A1 @ self.W2 + self.b2         # (n, C)
        y_hat = softmax(Z2)                  # (n, C)
        self.cache = {'X': X, 'Z1': Z1, 'A1': A1, 'Z2': Z2, 'y_hat': y_hat}
        return y_hat

    def backward(self, y_onehot):
        n = y_onehot.shape[0]
        X, Z1, A1, Z2, y_hat = (self.cache[k] for k in ('X','Z1','A1','Z2','y_hat'))

        # --- Gradient of loss w.r.t. Z2 (softmax + cross-entropy fused) ---
        # dL/dZ2 = (y_hat - y) / n
        dZ2 = (y_hat - y_onehot) / n         # (n, C)

        # --- Gradients for W2, b2 ---
        dW2 = A1.T @ dZ2                      # (h, C)
        db2 = dZ2.sum(axis=0)                 # (C,)

        # --- Backprop through ReLU ---
        dA1 = dZ2 @ self.W2.T                 # (n, h)
        dZ1 = dA1 * relu_grad(Z1)            # (n, h)  element-wise mask

        # --- Gradients for W1, b1 ---
        dW1 = X.T @ dZ1                       # (d, h)
        db1 = dZ1.sum(axis=0)                 # (h,)

        return {'W1': dW1, 'b1': db1, 'W2': dW2, 'b2': db2}

    def update(self, grads):
        self.W1 -= self.lr * grads['W1']
        self.b1 -= self.lr * grads['b1']
        self.W2 -= self.lr * grads['W2']
        self.b2 -= self.lr * grads['b2']

# --- Toy spiral dataset ---
def make_spiral(n_class=200, n_classes=2):
    X, y = [], []
    for c in range(n_classes):
        t = np.linspace(0, 1, n_class)
        angle = t * 3 * np.pi + (2 * np.pi * c / n_classes)
        r = 0.5 * t
        X.append(np.column_stack([r * np.cos(angle) + 0.05*np.random.randn(n_class),
                                   r * np.sin(angle) + 0.05*np.random.randn(n_class)]))
        y.extend([c] * n_class)
    return np.vstack(X), np.array(y)

X_sp, y_sp = make_spiral(n_class=200, n_classes=2)
n_classes = 2
y_onehot = np.eye(n_classes)[y_sp]

model = MLP(input_dim=2, hidden_dim=64, output_dim=n_classes, lr=0.1)
losses = []

for epoch in range(500):
    y_hat = model.forward(X_sp)
    loss = cross_entropy_loss(y_hat, y_onehot)
    grads = model.backward(y_onehot)
    model.update(grads)
    losses.append(loss)

print(f"Final loss: {losses[-1]:.4f}")
preds = model.forward(X_sp).argmax(axis=1)
print(f"Training accuracy: {np.mean(preds == y_sp):.3f}")


## Gradient Checking (Finite Differences)

The finite difference approximation: $\frac{\partial L}{\partial \theta_i} \approx \frac{L(\theta_i + \epsilon) - L(\theta_i - \epsilon)}{2\epsilon}$

If this doesn't match the analytical gradient to ~5+ decimal places, your backprop has a bug. This is the canonical sanity check — interviewers love asking if you know it.


In [ ]:
def numerical_gradient(model, X, y_onehot, param_name, eps=1e-5):
    """Compute numerical gradient for a single parameter tensor."""
    param = getattr(model, param_name)
    grad_num = np.zeros_like(param)
    it = np.nditer(param, flags=['multi_index'])
    while not it.finished:
        idx = it.multi_index
        old_val = param[idx]

        param[idx] = old_val + eps
        y_hat_plus = model.forward(X)
        loss_plus = cross_entropy_loss(y_hat_plus, y_onehot)

        param[idx] = old_val - eps
        y_hat_minus = model.forward(X)
        loss_minus = cross_entropy_loss(y_hat_minus, y_onehot)

        grad_num[idx] = (loss_plus - loss_minus) / (2 * eps)
        param[idx] = old_val
        it.iternext()
    return grad_num

# Use small model + small data for speed
model_small = MLP(input_dim=2, hidden_dim=4, output_dim=2, lr=0.1)
X_small = X_sp[:10]
y_small = y_onehot[:10]

y_hat = model_small.forward(X_small)
grads_analytic = model_small.backward(y_small)

for pname in ['W1', 'b1', 'W2', 'b2']:
    grad_num = numerical_gradient(model_small, X_small, y_small, pname)
    grad_ana = grads_analytic[pname]
    diff = np.max(np.abs(grad_num - grad_ana))
    rel  = diff / (np.max(np.abs(grad_num)) + np.max(np.abs(grad_ana)) + 1e-12)
    print(f"{pname}: max abs diff = {diff:.2e}, relative diff = {rel:.2e} {'✓' if rel < 1e-5 else '✗'}")


## Common Bugs Interviewers Plant

These are the actual bugs planted in "debug this" questions:


In [ ]:
# --- BUG 1: Wrong transpose ---
print("=== Bug 1: Wrong transpose in backprop ===")
# Correct: dW2 = A1.T @ dZ2   shape (h, C)
# Buggy:   dW2 = dZ2.T @ A1   shape (C, h) — wrong shape, crashes or silently corrupts

# --- BUG 2: Missing 1/n normalization ---
print("=== Bug 2: Missing 1/n in gradient ===")
# Correct: dZ2 = (y_hat - y) / n
# Buggy:   dZ2 = (y_hat - y)       — gradient is n times too large; LR must compensate

model_bug2 = MLP(input_dim=2, hidden_dim=64, output_dim=2, lr=0.1)
losses_bug2 = []
n = X_sp.shape[0]
for epoch in range(200):
    y_hat = model_bug2.forward(X_sp)
    loss = cross_entropy_loss(y_hat, y_onehot)
    losses_bug2.append(loss)
    # Buggy: no /n in backward
    dZ2 = (y_hat - y_onehot)          # BUG: should be / n
    dW2 = model_bug2.cache['A1'].T @ dZ2
    db2 = dZ2.sum(axis=0)
    dA1 = dZ2 @ model_bug2.W2.T
    dZ1 = dA1 * relu_grad(model_bug2.cache['Z1'])
    dW1 = X_sp.T @ dZ1
    db1 = dZ1.sum(axis=0)
    # Huge gradients → explode unless LR is tiny
    model_bug2.W2 -= 0.0001 * dW2  # need tiny LR to compensate
    model_bug2.b2 -= 0.0001 * db2

print(f"Bug2 model losses: {losses_bug2[0]:.4f} → {losses_bug2[-1]:.4f} (works only with tiny LR)")
print()

# --- BUG 3: Forgetting to not accumulate gradients (PyTorch context) ---
print("=== Bug 3: Gradient accumulation (PyTorch) ===")
print("In PyTorch: calling loss.backward() without optimizer.zero_grad()")
print("accumulates gradients across batches → effective gradient is sum of all steps")
print("Fix: always call optimizer.zero_grad() before loss.backward()")

# --- BUG 4: ReLU backward with wrong condition ---
print()
print("=== Bug 4: ReLU backward bug ===")
# Correct: (z > 0)    — zero exactly is 0 in gradient
# Buggy:   (z >= 0)   — technically also fine for practical purposes but inconsistent with convention
# Real bug: using the activated value A1 instead of pre-activation Z1
z_test = np.array([-1, 0, 1, 2])
print(f"Correct relu_grad (z > 0):  {(z_test > 0).astype(int)}")  # [0, 0, 1, 1]
print(f"Buggy   relu_grad using A1: {(relu(z_test) > 0).astype(int)}")  # same here but can differ


## Common Interview Questions

**Q: Derive the gradient of cross-entropy loss through softmax.**
Let $L = -\sum_i y_i \log(\hat{y}_i)$ and $\hat{y} = \text{softmax}(z)$. The combined gradient is $\partial L/\partial z_i = \hat{y}_i - y_i$. This simplification occurs because the derivative of the log-softmax has a particularly clean form when the loss is cross-entropy — that's why this pair is always used together.

**Q: Why do we transpose weight matrices in backprop?**
In the forward pass, $Z = AW$ maps $(n,h)\cdot(h,C)\to(n,C)$. The gradient flowing back is $dZ$ of shape $(n,C)$. To get $dA = dZ W^T$, we need $(n,C)\cdot(C,h)\to(n,h)$, so the transpose is required to reverse the shape transformation.

**Q: What is gradient checking and when do you use it?**
Finite difference approximation of the gradient: $(L(\theta+\epsilon) - L(\theta-\epsilon)) / 2\epsilon$. Agree with analytical gradient to ~5-6 decimal places? Backprop is correct. Use it during development on a tiny model/batch — it's O(parameters) in cost so too slow for production.

**Q: What is the vanishing gradient problem?**
With sigmoid/tanh activations, gradients shrink by the derivative of the activation at each layer. For sigmoid, max derivative is 0.25; after 10 layers, gradients can be $0.25^{10} \approx 10^{-6}$. ReLU fixes this by having gradient 1 for positive inputs, but introduces dying ReLU. Solutions: residual connections, careful init, batch norm, gradient clipping.

**Q: Why He initialization for ReLU?**
He init scales weights by $\sqrt{2/d_{in}}$, preserving activation variance through ReLU layers. Without this, activations shrink or explode through deep networks. Xavier init ($\sqrt{1/d_{in}}$) is designed for tanh/sigmoid, not ReLU.

## Key Takeaways
- Forward: $Z_1 = XW_1+b_1 \to A_1 = \text{ReLU}(Z_1) \to Z_2 = A_1 W_2 + b_2 \to \hat{y} = \text{softmax}(Z_2)$
- Backprop: gradient of cross-entropy + softmax is simply $(\hat{y} - y)/n$; chain rule + transposes for earlier layers
- Gradient check via finite differences: if relative error > 1e-5, there's a bug in backprop
- Top bugs: wrong transpose, missing /n, using A1 instead of Z1 for ReLU grad, forgetting zero_grad
- He init for ReLU: $\mathcal{N}(0, \sqrt{2/d_{in}})$; Xavier for tanh/sigmoid
- Gradient checking is the canonical correctness test — know it, explain it, implement it